In [1]:
from keras.src.losses import mean_squared_error
from sqlalchemy.dialects.mssql.information_schema import columns
from torch.backends.opt_einsum import strategy
%load_ext autoreload
%autoreload 2

import utils
from feature_engineering import FeatureEngineering
from sqlalchemy import create_engine, inspect, text
from sqlalchemy.types import Integer, Float, DateTime, REAL

import model
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import feature_engineering
import config  # все пути и настройки проекта лежат в config.py

ДАННЫЕ ДЛЯ ОБУЧЕНИЯ

In [2]:
np.random.seed(config.RANDOM_SEED)

users = np.random.choice(a=range(1, 140000), size=10000, replace=False)
data = pd.read_parquet(config.PARQUET_DIR,
                       filters=[('user_id', 'in', users)])

data.sort_values(by='ts', inplace=True)

X, y = data.drop(columns=['rating']), data['rating']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.75, shuffle=False)
X_valid, X_test, y_valid, y_test = train_test_split(X_valid, y_valid, train_size=0.5, shuffle=False)

In [3]:
X_train

,user_id,count,mean_rate,std_rate,min_rate,max_rate,movie_id,ratio_Action,ratio_Adventure,ratio_Animation,...,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,other,movie_cnt,av_rating_movie
1080712,68648,0,NaN,NaN,NaN,NaN,265,NaN,NaN,NaN,...,0,0,1,0,0,0,0,0,0,NaN
1080719,68648,1,5.000000,0.000000,5.0,5.0,296,0.000000,0.000000,0.000000,...,0,0,0,0,1,0,0,0,0,NaN
1080723,68648,2,5.000000,0.000000,5.0,5.0,315,0.000000,0.000000,0.000000,...,0,0,0,0,1,0,0,0,0,NaN
1080725,68648,3,3.666667,1.885618,1.0,5.0,319,0.333333,0.000000,0.000000,...,0,0,0,0,1,0,0,0,0,NaN
1080817,68648,8,4.125000,1.363589,1.0,5.0,593,0.375000,0.125000,0.000000,...,0,0,0,0,1,0,0,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1210039,8812,105,3.800000,0.667618,2.0,5.0,8961,0.571429,0.409524,0.057143,...,0,0,0,0,0,0,0,0,11951,3.979249
1210066,8812,106,3.801887,0.664743,2.0,5.0,50872,0.575472,0.415094,0.066038,...,0,0,0,0,0,0,0,0,2857,3.938047
1209943,8812,107,3.799065,0.662267,2.0,5.0,2762,0.570093,0.411215,0.074766,...,0,1,0,0,0,0,0,0,29356,4.112311
1210018,8812,108,3.805556,0.662603,2.0,5.0,7090,0.564815,0.407407,0.074074,...,0,0,0,0,0,0,0,0,5232,3.926510


ОБУЧЕНИЕ БАЗОВОЙ МОДЕЛИ

In [4]:
import model

base_model = model.Model(
    model=model.BaseModel,
    strategy='mean',
    config_value=4.0
).fit(X_train, y_train)

print(base_model.get_metrics(X_train, y_train))
print(base_model.get_metrics(X_valid, y_valid))
print(base_model.get_metrics(X_test, y_test))

{'MSE': 0.9403315599736105, 'MAE': 0.762138985362344, 'MAPE': 0.3284829795543505}
{'MSE': 1.201139382038556, 'MAE': 0.7987418685244294, 'MAPE': 0.4271952952438355}
{'MSE': 1.156006444079518, 'MAE': 0.7764144892980502, 'MAPE': 0.40494581449095735}


ТЮНИНГ

In [ ]:
from sklearn.metrics import mean_squared_error as MSE
from catboost import CatBoostRegressor

tun = model.Tuner(
    n_trials=50,
    direction='minimize',
    metric=MSE,
    model=CatBoostRegressor
)
F = config.FEATS_CATBOOST
tun.tuning(x_train=X_train[F], y_train=y_train, x_valid=X_valid[F], y_valid=y_valid, save_mode=True)

[I 2026-11-13 03:57:23,083] A new study created in memory with name: find_hp
[I 2026-11-13 03:57:43,623] Trial 0 finished with value: 0.6856993297253938 and parameters: {'depth': 6, 'learning_rate': 0.2678154091306088, 'l2_leaf_reg': 0.8471801418819978}. Best is trial 0 with value: 0.6856993297253938.
[I 2026-11-13 04:00:19,429] Trial 1 finished with value: 0.67883959374282 and parameters: {'depth': 8, 'learning_rate': 0.04296748115469074, 'l2_leaf_reg': 0.004207053950287938}. Best is trial 1 with value: 0.67883959374282.
[I 2026-11-13 04:01:07,417] Trial 2 finished with value: 0.6852272679264957 and parameters: {'depth': 4, 'learning_rate': 0.22044355215811345, 'l2_leaf_reg': 0.2537815508265665}. Best is trial 1 with value: 0.67883959374282.


ОБУЧЕНИЕ БУСТИНГА (CATBOOST)

In [10]:
import catboost

cat_model = model.Model(
    model=catboost.CatBoostRegressor,
    iterations=25,
    verbose=False,
    learning_rate=0.1,
    depth=7
).fit(X_train[config.FEATS_CATBOOST], y_train)

print(cat_model.get_metrics(X_train[config.FEATS_CATBOOST], y_train))
print(cat_model.get_metrics(X_valid[config.FEATS_CATBOOST], y_valid))
print(cat_model.get_metrics(X_test[config.FEATS_CATBOOST], y_test))


{'MSE': 0.780088808783664, 'MAE': 0.6881724009960496, 'MAPE': 0.2943816511816415}
{'MSE': 0.7349708919017053, 'MAE': 0.6565325186596256, 'MAPE': 0.3061521491697829}
{'MSE': 0.7396555854367493, 'MAE': 0.6505757357111909, 'MAPE': 0.30206110469356157}


In [6]:
d = pd.DataFrame(
    {'a':[1], 'b':[0]}
)


print(d.iloc[0].to_dict())

{'a': 1, 'b': 0}
